# Stand & Walk: PPO locomotion policies for Metal2Humanoid embodiments

**Metal2Humanoid · Models → Learning**

In this project you take the robot embodiments from the course (Asimov, Berkeley Humanoid Lite, Atom01, the Frankensteins) and teach them to **stand** and **walk** with reinforcement learning, in a way that is meant to transfer to the real hardware later:

* the policy commands **joint position targets** that a PD controller turns into torques, exactly like the motor drivers on a real humanoid;
* the policy only sees what a real robot can measure: IMU angular velocity, gravity direction, joint angles/velocities, its own last action and a velocity command;
* physics is MuJoCo, running hundreds of robots in parallel on your CPU; the neural network trains on the GPU (or CPU, it is small).

Everything lives in the `standwalk` package next to this notebook:

| file | what it does |
|---|---|
| `standwalk/embodiments.py` | registry of robots: XML, base body, feet, joint groups, default pose, PD gains |
| `standwalk/env.py` | `HumanoidVecEnv`: N parallel MuJoCo worlds, observations, rewards, resets |
| `standwalk/ppo.py` | actor-critic, PPO, checkpoints, policy export for the robot |
| `standwalk/render.py` | offscreen rendering and videos |
| `train.py` / `evaluate.py` | command-line entry points for long runs |

Run the cells top to bottom. The in-notebook training takes a few minutes; the long runs are launched from the command line and loaded back here.

## 0. Setup

Local: create a virtualenv and `pip install -r requirements.txt` (MuJoCo ≥ 3.1, PyTorch, numpy, imageio, matplotlib). The embodiments come from the `metal2humanoid` repository, expected next to this notebook (or set `STANDWALK_EMBODIMENTS`).

Google Colab: run the next cell, it clones both repositories and installs the dependencies. A CPU runtime is fine for this notebook; the long walking run is better on a machine with more cores.

In [ ]:
import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("standwalk").exists():
        subprocess.run("git clone https://github.com/hbrnetwork/metal2humanoid.git", shell=True, check=True)
        # TODO: replace with the URL of the repository that contains this notebook and the standwalk package
        subprocess.run("pip install -q mujoco imageio imageio-ffmpeg", shell=True, check=True)
os.environ.setdefault("MUJOCO_GL", "egl" if sys.platform.startswith("linux") and not os.environ.get("DISPLAY") else os.environ.get("MUJOCO_GL", ""))
if not os.environ["MUJOCO_GL"]:
    del os.environ["MUJOCO_GL"]
import warnings; warnings.filterwarnings("ignore")
import time, json, math
import numpy as np, torch, matplotlib.pyplot as plt, mujoco
from IPython.display import display, HTML, Video
import standwalk
from standwalk.embodiments import REGISTRY, load_embodiment, describe, embodiments_dir
from standwalk.env import HumanoidVecEnv, TaskConfig, stand_config, walk_config, DEFAULT_REWARDS
from standwalk.ppo import PPOConfig, train, Policy, export_policy, load_checkpoint
from standwalk.render import EnvRenderer, rollout_video, save_video
print("mujoco", mujoco.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cpus", os.cpu_count())
print("embodiments:", embodiments_dir())

## 1. The embodiments

Every robot is a MuJoCo XML. The registry (`standwalk/embodiments.py`) records, per robot, which body is the floating base, which bodies are the feet, which joints are actuated, a **default pose** (slightly bent knees) and **PD gains** per joint group. Loading an embodiment rewrites its XML so that all robots share one control interface, then *settles* it: the robot is placed on the floor in its default pose and held there by the PD controller for 1.5 s.

In [ ]:
loaded = {}
print(f"{'embodiment':24s} {'DoF':>4s} {'mass':>8s} {'standing h':>11s} {'tilt after PD hold':>19s}  stands on its own?")
for name in REGISTRY:
    le = load_embodiment(name); loaded[name] = le
    print(f"{name:24s} {le.n_act:4d} {le.total_mass:7.1f}kg {le.nominal_height:10.3f}m {le.settle_tilt_deg:18.1f}°  {'yes' if le.stands else 'no'}")

In [ ]:
fig, axes = plt.subplots(1, len(REGISTRY), figsize=(3.2 * len(REGISTRY), 3.4))
for ax, name in zip(axes, REGISTRY):
    env = HumanoidVecEnv(loaded[name], num_envs=1, task=stand_config(init_joint_noise=0.0, init_yaw_random=False))
    ax.imshow(EnvRenderer(env, 400, 400).frame(0)); ax.set_title(name, fontsize=9); ax.axis("off"); env.close()
plt.tight_layout(); plt.show()

Print the joint table of one robot. `swing` marks the leg pitch joints (hip pitch, knee, ankle pitch) that are free to move when walking; all other joints get a small penalty for leaving the default pose. `tau` is the torque limit the PD controller saturates at.

In [ ]:
print(describe(loaded["asimov"]))

## 2. Control: position targets and a PD controller

Real humanoid motors are driven by a high-rate PD loop: $\tau = k_p\,(q^{*} - q) - k_d\,\dot q$, clipped to the motor's torque limit. The learned policy runs at 50 Hz and only chooses the target $q^{*}$:

$$ q^{*} = q_{\text{default}} + s \cdot a, \qquad a = \pi(o) \in \mathbb{R}^{n_{\text{joints}}},\quad s = 0.25\ \text{rad} $$

MuJoCo implements the PD law natively with `<position kp kv>` actuators, evaluated at the physics rate (4 ms), so what the policy "feels" in simulation is the same closed-loop joint behaviour the robot will have.

**Why PD alone is not enough.** A pure PD hold of the default pose is an open-loop posture: if the body leans, gravity pulls it further, and only the ankle stiffness resists. The table above shows that only the lightest robot stays up. Let us look at a push test with zero policy action:

In [ ]:
def pd_hold_push_test(name, push_vel=0.3, seconds=4.0):
    env = HumanoidVecEnv(loaded[name], num_envs=1, task=stand_config(init_joint_noise=0.0, init_yaw_random=False,
                                                                    push_interval_seconds=1.0, push_max_vel=push_vel))
    env.reset(); tilt = []
    for t in range(int(seconds / env.dt)):
        *_, term, trunc, info = env.step(np.zeros((1, env.act_dim), np.float32))
        tilt.append(math.degrees(math.acos(-float(np.clip(env.proj_grav[0, 2], -1, 1)))))
        if term[0]: break
    env.close(); return np.array(tilt)

plt.figure(figsize=(8, 3.5))
for name in REGISTRY:
    t = pd_hold_push_test(name); plt.plot(np.arange(len(t)) * 0.02, t, label=name)
plt.axhline(50, ls="--", c="k", lw=0.8); plt.xlabel("time [s]"); plt.ylabel("base tilt [deg]")
plt.title("PD hold of the default pose, random 0.3 m/s pushes every second (dashed = termination)"); plt.legend(fontsize=8); plt.show()

## 3. The environment

`HumanoidVecEnv` steps N robots in lock-step. One call to `env.step(actions)` advances every robot by one control period (5 physics substeps) using MuJoCo's multithreaded `rollout`, and returns:

* `obs` for the actor, `critic_obs` for the critic (same plus privileged state), `reward`, `terminated` (fell), `truncated` (20 s time limit) and an `info` dict.

### Observations (what the real robot can measure)

In [ ]:
env = HumanoidVecEnv(loaded["asimov"], num_envs=4, task=walk_config(), seed=0)
obs, cobs = env.reset()
print("actor obs dim", env.obs_dim, " critic obs dim", env.critic_obs_dim, " action dim", env.act_dim)
print("\nactor observation layout:")
for n, s in env.obs_layout: print(f"  {n:26s} {s}")
print("critic extras:", env.critic_extra_layout)
print("\nobservation scales: ang vel x%.2f, joint vel x%.2f, command x%s" % (env.cfg.scale_ang_vel, env.cfg.scale_dof_vel, env.cfg.scale_cmd))

Note what is **not** in the actor's observation: the base position, the base linear velocity, foot contacts. Those need external sensing or a state estimator on a real robot, so we only give them to the critic (asymmetric actor-critic): the critic trains a better value function, the actor stays deployable.

### Rewards

The reward is a weighted sum of named terms. Weights are **per second** (multiplied by the 0.02 s control period) so they do not depend on the control rate; only the fall penalty is a one-off. The default weights follow legged_gym / IsaacLab practice:

In [ ]:
groups = {"task": ["tracking_lin_vel", "tracking_ang_vel", "alive"],
          "gait shaping": ["feet_air_time", "gait_contact", "feet_swing_height"],
          "regularisation": ["lin_vel_z", "ang_vel_xy", "orientation", "base_height", "torques", "dof_acc", "dof_vel",
                             "action_rate", "posture", "legs_pose", "dof_pos_limits", "feet_slip"],
          "one-off": ["termination"]}
stand, walk = stand_config().rewards, walk_config().rewards
print(f"{'term':20s} {'walk':>8s} {'stand':>8s}")
for g, names in groups.items():
    print(f"-- {g}")
    for n in names: print(f"{n:20s} {walk[n]:8.3g} {stand[n]:8.3g}")

* `tracking_lin_vel` / `tracking_ang_vel`: $\exp(-\|v_{cmd}-v\|^2/\sigma)$, the actual task.
* `gait_contact` and `feet_swing_height` use a **gait clock** (phase $\phi$ advancing at 1.5 Hz; left foot swings when $\sin\phi>0$, right foot half a cycle later). The clock is also in the observation. When the command is zero the schedule says "both feet down".
* `feet_air_time` rewards long swings at touchdown, `feet_slip` penalises foot velocity while in contact.
* the rest keeps motion smooth and realistic (torques, accelerations, action rate, posture) so that it survives the sim-to-real gap.

**Stand is walk with a zero velocity command** plus a pull of the leg joints towards the default pose. Same observation and action space, so a stand policy can initialise a walk policy.

### Episodes
Each episode starts in the default pose with small joint noise and a random heading, lasts 20 s, and ends early if the base tilts more than 50°, the CoM drops below half its nominal height, or the base touches the floor. Commands are resampled every 5 s; 20 % of them are "stand still".

In [ ]:
# what a random policy experiences: per-term contributions over a few episodes
rng = np.random.default_rng(0)
for t in range(400):
    env.step(rng.normal(0, 0.5, (env.N, env.act_dim)).astype(np.float32))
stats = env.pop_stats(); env.close()
terms = {k[4:]: v for k, v in stats.items() if k.startswith("rew/")}
plt.figure(figsize=(9, 3)); plt.bar(terms.keys(), terms.values()); plt.xticks(rotation=60, ha="right", fontsize=8)
plt.title(f"random policy: episode return {stats['return']:.2f}, mean length {stats['length']:.0f} steps, fall rate {stats['fall_rate']:.2f}"); plt.show()

## 4. PPO, and training a stand policy

The trainer in `standwalk/ppo.py` is the locomotion-PPO recipe (rsl_rl / legged_gym) in ~200 lines:

1. **Rollout**: run the current policy for 24 steps in all N environments (N·24 samples), storing observations, actions, log-probabilities and values.
2. **Advantages** with GAE ($\gamma=0.99$, $\lambda=0.95$); time-limit truncations are bootstrapped with $V(s_T)$ so a 20 s episode is not punished like a fall.
3. **Update**: 5 epochs over 4 minibatches of the clipped surrogate objective (clip 0.2), a clipped value loss and a small entropy bonus.
4. **Adaptive learning rate**: the KL divergence between the rollout policy and the updated policy is measured; too large → learning rate / 1.5, too small → × 1.5.
5. Running mean/std normalisation of observations (saved with the policy), separate actor and critic MLPs (512-256-128, ELU), Gaussian policy with a learned standard deviation.

A few minutes are enough for the Asimov legs to learn to stand. We train with 128 environments for 2 M steps (use the command line for anything longer):

In [ ]:
RUN_STAND = "runs/notebook_asimov_stand"
env = HumanoidVecEnv(loaded["asimov"], num_envs=128, task=stand_config(), seed=0)
cfg = PPOConfig(total_steps=2_000_000, save_every=20, log_every=25)
t0 = time.time()
rows = train(env, cfg, RUN_STAND)
env.close()
print(f"trained in {(time.time()-t0)/60:.1f} min")

In [ ]:
def plot_progress(rows, title):
    it = [r["iteration"] for r in rows if "return" in r]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].plot(it, [r["return"] for r in rows if "return" in r]); ax[0].set_title("episode return"); ax[0].set_xlabel("iteration")
    ax[1].plot(it, [r["length"] for r in rows if "return" in r]); ax[1].set_title("episode length [steps]"); ax[1].set_xlabel("iteration")
    ax[2].plot(it, [r["fall_rate"] for r in rows if "return" in r]); ax[2].set_title("fall rate"); ax[2].set_xlabel("iteration")
    fig.suptitle(title); plt.tight_layout(); plt.show()
plot_progress(rows, "Asimov stand (in-notebook run)")

### Evaluate: video and a push test

`Policy` loads a checkpoint (network + observation statistics) and maps observations to the **mean** action. We record one robot for 8 s and then test robustness to pushes it has never seen in training.

In [ ]:
def show_video(path, frames, every=1.0, dt=0.02, n=6):
    idx = np.linspace(0, len(frames) - 1, n).astype(int)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.4))
    for ax, i in zip(axes, idx): ax.imshow(frames[i]); ax.set_title(f"t = {i*dt:.1f} s", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()
    if path is not None and str(path).endswith(".mp4"): display(Video(str(path), embed=True, width=480))

pol = Policy(f"{RUN_STAND}/best.pt")
ev = HumanoidVecEnv(loaded["asimov"], num_envs=1, task=stand_config(), seed=7, n_threads=1)
frames, stats, path = rollout_video(ev, pol, seconds=8.0, command=[0, 0, 0], path=f"{RUN_STAND}/stand.mp4")
ev.close(); print(stats); show_video(path, frames)

In [ ]:
from evaluate import evaluate
for push in (0.0, 3.0):
    task = stand_config(push_interval_seconds=push, push_max_vel=0.5)
    ev = HumanoidVecEnv(loaded["asimov"], num_envs=32, task=task, seed=11)
    m = evaluate(pol, ev, seconds=10.0, cmd=[0, 0, 0]); ev.close()
    print(f"pushes every {push or 'never':>5} s: survival {m['mean_survival_s']:5.1f} s  fall rate {m['fall_rate']:.2f}  mean |tau| {m['mean_abs_torque_nm']:.1f} Nm")

If the fall rate under pushes is high, that is expected: nothing in training asked for it. Training with `push_interval_seconds=3` (`--push-interval 3` on the command line) is the first thing to try, and it is one of the assignment tasks.

## 5. Walking

Walking is the same environment with velocity commands $(v_x, v_y, \dot\psi)$ sampled from $[-0.4, 0.8]$ m/s, $[-0.3, 0.3]$ m/s and $[-0.8, 0.8]$ rad/s. It needs on the order of $10^8$ environment steps. From the command line, on a machine with 8 cores this is one to three hours:

```bash
python train.py --embodiment asimov --task walk --num-envs 512 --total-steps 120e6 --out runs/asimov_walk --video
python evaluate.py --checkpoint runs/asimov_walk/best.pt --cmd 0.5 0 0 --video videos/asimov_walk.mp4
```

Below we load that run if it exists (progress is in `progress.csv`, the best policy in `best.pt`).

In [ ]:
import csv
RUN_WALK = "runs/asimov_walk"
def read_progress(run):
    with open(f"{run}/progress.csv") as f:
        rows = []
        for r in csv.DictReader(f):
            rows.append({k: (float(v) if v not in ("", None) else np.nan) for k, v in r.items()})
    return rows
if os.path.exists(f"{RUN_WALK}/progress.csv"):
    wrows = read_progress(RUN_WALK)
    plot_progress(wrows, f"Asimov walk ({wrows[-1]['total_steps']/1e6:.0f} M steps)")
    terms = [k for k in wrows[-1] if k.startswith("rew/")]
    plt.figure(figsize=(10, 3.2))
    for k in ["rew/tracking_lin_vel", "rew/tracking_ang_vel", "rew/feet_air_time", "rew/gait_contact", "rew/feet_swing_height", "rew/orientation", "rew/action_rate", "rew/torques"]:
        plt.plot([r["iteration"] for r in wrows], [r[k] for r in wrows], label=k[4:])
    plt.legend(fontsize=7, ncol=4); plt.xlabel("iteration"); plt.title("per-term episode contributions during training"); plt.show()
else:
    print("no walk run found yet: launch train.py as shown above")

In [ ]:
if os.path.exists(f"{RUN_WALK}/best.pt"):
    wpol = Policy(f"{RUN_WALK}/best.pt")
    print("walk policy trained for", wpol.ckpt["meta"]["total_steps"], "steps")
    for cmd in ([0.5, 0, 0], [0, 0, 0], [0.3, 0, 0.5], [-0.3, 0, 0], [0, 0.3, 0]):
        ev = HumanoidVecEnv(loaded["asimov"], num_envs=16, task=walk_config(), seed=3)
        m = evaluate(wpol, ev, seconds=10.0, cmd=cmd); ev.close()
        print(f"cmd vx={cmd[0]:+.1f} vy={cmd[1]:+.1f} yaw={cmd[2]:+.1f}: survival {m['mean_survival_s']:5.1f}s  fall rate {m['fall_rate']:.2f}  velocity error {m['mean_vel_error_mps']:.2f} m/s")
    ev = HumanoidVecEnv(loaded["asimov"], num_envs=1, task=walk_config(), seed=5, n_threads=1)
    frames, stats, path = rollout_video(ev, wpol, seconds=8.0, command=[0.5, 0, 0], path=f"{RUN_WALK}/walk_0.5.mp4"); ev.close()
    print(stats); show_video(path, frames)

## 6. All embodiments: results gallery

Nothing above was specific to Asimov: the embodiment name selects the robot, the registry entry supplies the rest. The cell below collects every run directory produced by `train.py` (`runs/<embodiment>_<task>`), shows the learning curves, a strip of frames from the end-of-training video, and re-evaluates each walk policy on a forward command. Runs that do not exist yet are simply skipped, so this section fills in as the training queue progresses.

In [ ]:
import glob, re
RUN_NAMES = {"asimov": "asimov", "bhl": "berkeley_humanoid_lite", "bhl_noarms": "berkeley_humanoid_lite_noarms", "frankenstein_v1": "frankenstein_v1",
             "frankenstein_v0": "frankenstein_v0", "atom01": "atom01", "mjc_humanoid": "mjc_humanoid"}
def run_embodiment(run_dir):
    m = re.match(r"(.+)_(stand|walk)$", os.path.basename(run_dir))
    return (RUN_NAMES.get(m.group(1)), m.group(2)) if m else (None, None)

runs = sorted(d for d in glob.glob("runs/*_stand") + glob.glob("runs/*_walk")
              if os.path.exists(f"{d}/progress.csv") and run_embodiment(d)[0] and "notebook" not in d)
summary = []
for d in runs:
    emb, task = run_embodiment(d)
    rows_ = read_progress(d); last = rows_[-1]
    tail = rows_[-50:]
    rec = {"run": os.path.basename(d), "embodiment": emb, "task": task, "steps (M)": last["total_steps"] / 1e6,
           "return (last 50 it)": np.nanmean([r["return"] for r in tail]), "fall rate": np.nanmean([r["fall_rate"] for r in tail]),
           "train min": last["time_s"] / 60}
    if os.path.exists(f"{d}/best.pt"):
        pol_ = Policy(f"{d}/best.pt")
        cmd = [0.5, 0, 0] if task == "walk" else [0, 0, 0]
        ev = HumanoidVecEnv(loaded[emb], num_envs=8, task=TaskConfig(**pol_.ckpt["task_config"]), seed=21)
        m_ = evaluate(pol_, ev, seconds=10.0, cmd=cmd); ev.close()
        rec.update({"eval cmd": cmd, "eval fall rate": m_["fall_rate"], "vel err (m/s)": m_["mean_vel_error_mps"]})
    summary.append(rec)
import pandas as pd
display(pd.DataFrame(summary).round(3))

In [ ]:
import imageio.v2 as imageio
for d in runs:
    emb, task = run_embodiment(d)
    rows_ = read_progress(d)
    fig = plt.figure(figsize=(16, 2.6)); gs = fig.add_gridspec(1, 8)
    ax = fig.add_subplot(gs[0, :2]); it_ = [r["iteration"] for r in rows_]
    ax.plot(it_, [r["return"] for r in rows_], lw=0.8); ax.set_title(f"{os.path.basename(d)}: return", fontsize=9); ax.set_xlabel("iteration")
    ax2 = fig.add_subplot(gs[0, 2]); ax2.plot(it_, [r["fall_rate"] for r in rows_], lw=0.8, c="C3"); ax2.set_title("fall rate", fontsize=9); ax2.set_ylim(-0.02, 1.02)
    vid = f"{d}/video.mp4"
    if os.path.exists(vid):
        fr = imageio.mimread(vid, memtest=False)
        for k, i in enumerate(np.linspace(0, len(fr) - 1, 5).astype(int)):
            a = fig.add_subplot(gs[0, 3 + k]); a.imshow(fr[i]); a.set_title(f"t={i/50:.1f}s", fontsize=8); a.axis("off")
    plt.tight_layout(); plt.show()

Things to look for in the gallery: the stand runs should reach zero fall rate within a few million steps; the walk runs take longer and their return keeps improving as the gait becomes smoother (the regularisation terms shrink). A robot that stands but has a high velocity error when walking has usually learned to shuffle or to stand through the command; the per-term columns of `progress.csv` (`rew/feet_air_time`, `rew/feet_swing_height`) tell you which.

## 7. Towards the real robot

`evaluate.py --export` (or `export_policy`) writes two files for the robot's onboard computer:

* `<name>.pt`: a TorchScript module `obs -> action` with the observation normalisation folded in;
* `<name>.json`: the observation layout and scales, joint order, default pose, PD gains, torque limits, action scale and control period.

The robot-side loop is then:

```
every 20 ms:
    obs = [cmd*scale, gyro*0.25, gravity_in_body_frame, q - q_default, qd*0.05, last_action, sin(phase), cos(phase)]
    a   = policy(obs)
    q*  = q_default + 0.25 * a          -> send as position targets to the motor drivers (kp, kd from the json)
```

In [ ]:
out = export_policy(f"{RUN_STAND}/best.pt", f"{RUN_STAND}/asimov_stand_policy")
meta = json.load(open(str(out).replace(".pt", ".json")))
print("exported", out, "\njoint order:", meta["joint_names"][:4], "...\nobs layout:", meta["obs_layout"])
scripted = torch.jit.load(str(out)); print("TorchScript test action:", scripted(torch.zeros(1, meta["obs_dim"])).shape)

What still separates this policy from the hardware (and what the assignment's stretch tasks are about):

| gap | what to do in simulation |
|---|---|
| sensor noise and bias | `obs_noise=1.0` in `TaskConfig` (`--obs-noise 1`) |
| disturbances, uneven floors | `push_interval_seconds`, terrain (not implemented) |
| actuator limits, latency | torque limits are enforced; add an action delay of 1–2 control steps |
| wrong masses / friction | per-env model randomisation: `mujoco.rollout` accepts one model per env |
| PD gains | they must match the gains configured on the real motor drivers; the registry values are *guesses* |
| contact geometry | the collision shapes in the XML (feet!) must match the real robot's |

## 8. The assignment

Deliverables: your modified code, the run directories (`progress.csv`, `best.pt`) and a short report (`.mdx` note under `material/notes/rl/` following the course template) with curves, videos and what you learned.

1. **Stand, two robots (30 %).** Train stand policies for Asimov and one other embodiment (10 M steps each). Report return/length/fall-rate curves, a video, and the per-term reward breakdown at convergence. Explain which terms dominate and why.
2. **Robustness (20 %).** Evaluate your stand policies under pushes (`--push-interval 3`) and observation noise (`--obs-noise 1`). Then retrain *with* those perturbations and compare. One table, one paragraph.
3. **Walk (30 %).** Train a walking policy for one embodiment (≥ 100 M steps). Evaluate forward, backward, sideways and turning commands; report velocity tracking error and fall rate for each. Include a video. Try at least one reward change (e.g. gait frequency, `feet_air_time` weight, `action_rate`) and explain the effect on the gait.
4. **Sim-to-real contract (20 %).** Export the policy and write the robot-side observation and control code (pseudo-code is fine) for your robot's actual sensors and motor driver API. List every assumption the policy makes about the hardware (gains, joint order, sign conventions, control rate).
5. **Stretch (bonus).** Add an embodiment to the registry (`REGISTRY` in `standwalk/embodiments.py`), implement per-env mass/friction randomisation with a list of models, or add an action-delay buffer and show the policy still walks.

Grading looks for correct, reproducible experiments and honest analysis, not for the highest return.

## Appendix A. Adding an embodiment

```python
REGISTRY["my_robot"] = Embodiment(
    name="my_robot", xml="my-robot/xmls/my_robot.xml",
    base_body="pelvis", feet=("left_foot", "right_foot"),
    pd=((r"hip|knee", 80.0, 2.5, 60.0), (r"ankle", 40.0, 1.5, 30.0), (r"arm", 20.0, 0.5, 20.0)),
    default_pose={r"hip_pitch": -0.2, r"knee": 0.4, r"ankle_pitch": -0.2},
    swing_joints=r"(hip_pitch|knee|ankle_pitch)")
```
Requirements on the XML: a free joint on the base body, named hinge joints, collision geoms on the feet, meshes reachable from `meshdir`. Check it with `python -m standwalk.tools check my_robot -v`; the settle test tells you whether the default pose and gains make sense (watch for the right knee sign on mirrored models).

## Appendix B. Tuning notes

* **It falls immediately and never improves**: PD gains too low/high (check `tools check`), default pose unstable, or torque limits too small. Try the stand task first.
* **It stands but never walks**: increase `feet_air_time` / `gait_contact`, lower `action_rate` and `torques` penalties, check that `gait_freq` is sensible for the leg length (shorter legs → faster), train longer.
* **Jittery motion**: raise `action_rate`, `dof_acc`; lower the learning rate.
* **KL / learning rate**: the trainer logs `kl` and `lr`. Healthy runs sit around KL 0.01, lr 1e-4 … 5e-4.
* **Speed**: `python -m standwalk.tools bench` measures env steps/s. More envs help up to ~4× the number of cores; the GPU only matters for the network update.